# flexible neutrino oscillation simulator 

(c) Isabel Goos, Yael Deniz, Estelle Salomé, Mahé Lucas, Baptiste Lamy, Nobuaki Fuji

15/07/2026



## loading packages

In [ ]:
# Locate flexOPT securely without relying on @__DIR__ (unreliable in IJulia).
# If this notebook is outside the repository, set ENV["FLEXOPT_ROOT"] first.
import Pkg

function find_flexopt_root(start_dir=pwd())
    candidates = String[]
    if haskey(ENV, "FLEXOPT_ROOT")
        push!(candidates, abspath(expanduser(ENV["FLEXOPT_ROOT"])))
    end
    directory = abspath(start_dir)
    while true
        push!(candidates, directory)
        parent = dirname(directory)
        parent == directory && break
        directory = parent
    end
    for candidate in unique(candidates)
        project_file = joinpath(candidate, "Project.toml")
        source_dir = joinpath(candidate, "src")
        if isfile(project_file) && isfile(joinpath(source_dir, "commonBatchs.jl"))
            return candidate
        end
    end
    error("Cannot locate flexOPT. Start Jupyter inside the repository or set ENV[\"FLEXOPT_ROOT\"] to its absolute path.")
end

flexopt_root = find_flexopt_root()
Pkg.activate(flexopt_root)
@show VERSION Threads.nthreads() Base.active_project()


In [ ]:

include(joinpath(flexopt_root, "src", "commonBatchs.jl"))
include(joinpath(flexopt_root, "src", "planet1D.jl"))
planet1D.configure_input!()
include(joinpath(flexopt_root, "src", "GeoPoints.jl"))

include(joinpath(flexopt_root, "src", "neutrinoOscillation.jl"))


using .commonBatchs, .planet1D, .GeoPoints
using .neutrinoOscillation

In [ ]:
using CairoMakie, LaTeXStrings

In [ ]:
include(joinpath(flexopt_root, "src", "neutrinoFlux.jl"))
using .neutrinoFlux

## local data (every user should control this)

In [ ]:
localConfig = joinpath(@__DIR__, "neutrinoPropagation_local.jl")
isfile(localConfig) || error("Copy neutrinoPropagation.config.example.jl to neutrinoPropagation_local.jl and set your local geodynamical files.")
include(localConfig)
all(isfile, geody_files) || error("At least one file configured in neutrinoPropagation_local.jl does not exist.")

## constants

## geodynamical models (if we want to)

In [ ]:
ρGeodyFile = geody_files.density
compositionGeodyFile = geody_files.composition
temperatureGeodyFile = geody_files.temperature
wGeodyFile = geody_files.water

### binning of energies (binned logarithmically) and cos \theta (binned linearly)

In [ ]:
numberEnergyBins = 50
minEnergy = 1.0
maxEnergy = 100.0

numberAngleBins = 100
mincosθ = -1
maxcosθ = 0

azimuthDisk = 0.0 # for 2d disk 0.0 or 180.0

energies = logrange(minEnergy,maxEnergy,numberEnergyBins)
cosθgrid = range(mincosθ,maxcosθ,numberAngleBins)
binning = (energies=energies, cosθgrid=cosθgrid)
# makeArrayBins(binningEnergy; option="linear") something like this should be made

# computeFlux(binning) # honda/daemon is already decided by DEFAULT_FLUX_TABLE; 

# maybe computeFlux(binning::this binnning Tuple)  = computeFlux(makeArrayBins(binningEnergy; option="log"))
# or for the custom arrays (energy array, cosarray ) <- bigger than binningArray (6 components)

# configurations (Earth model, flux model and detector model)

The background 1D Earth model is configured directly with `planet1D.configure_input!()`.
Use `planet1D.getSet1Dmodel!` to select an Earth, Mars, Moon, or custom model.


Another remark is that I prefer to rotate the virtual Earth based on the north pole definition (if we set detector position as the reference, it might be a bit of tricky for multi-detector situations)

### flavours

In [ ]:
osc = OscillationParameters(3)
setθ!(osc, 1=>2, 0.59)
setθ!(osc, 1=>3, 0.15)
setθ!(osc, 2=>3, 0.84)
setδ!(osc, 1=>3, 3.86)
setΔm²!(osc, 2=>3, -2.523e-3)
setΔm²!(osc, 1=>2, -7.39e-5)
U = PMNSMatrix(osc)
H = Hamiltonian(osc)
Uround = roundExt.(U, 0.01)
Hround = roundExt.(H, 0.00001)


### 1D Earth configuration

In [ ]:
#change_default_flux!(:Honda)

In [ ]:
set_default_planet!(:Earth) # :Earth applies WGS84 (otherwise use :SphericalEarth)
configure_input!()
getSet1Dmodel!(:PREM) # PREM is default
topographyPrecisionKm = 5.0 # if you would like to work on the topography 

### detector point 

NB: the detectorPoint's longitude will be ignored if we work with 2D Earth disk (for the sake of rotation)

In [ ]:
detectorPoint = GeoPoint(36.296761,15.978403;alt=-2500.0) # lat, lon with altitude in metre KM3NeT ORCA 

### rotation of north poles from real to virtual

In [ ]:
northPolePoint = GeoPoint(90.0,0.0) 
northPoleInVirtualEarth = GeoPoint(30.0,0.0) # Another proposition of rotation, if we are working with 2D disk, the longitude should be 0 or 180.0 (on the left half point)

In [ ]:
# Construct a regular 2D Cartesian disk through the real ellipsoidal Earth.
# `centreOfPlanet` keeps the disk origin at the planet centre.
Δx = 5e3 # m
Δz = 5e3 # m
altMin, altMax = -6400e3, 6400e3
horizontalMin, horizontalMax = -6400e3, 6400e3

boxGrids = constructLocalBox(
    northPolePoint,
    Δx,
    Δz,
    altMin,
    altMax,
    horizontalMin,
    horizontalMax;
    centreOption="centreOfPlanet",
    axis_angle_deg=0.0,
)
# axis_angle_deg=0 selects a south-to-north meridional cut.


### Prepare the ellipsoidal seismic model, including real topography

`allGridsInCartesian` remains a regular numerical grid. Topography changes the material assigned to each point, not these coordinates.


In [ ]:

seismicCacheName = "seismicModel2D_topography_$(topographyPrecisionKm)km"
seismicModel = lazyProduceOrLoad(
    seismicCacheName,
    getParamsAndTopo,
    boxGrids.allGridsInGeoPoints,
    copy(boxGrids.effectiveRadii),
    topographyPrecisionKm,
)


### Prepare the virtual spherical-Earth coordinates

`convertedGrids` is useful for geographic diagnostics. `effectiveCoords` is the separate, generally non-separable Cartesian lookup grid used to sample the spherical StagYY model. The regular `boxGrids.allGridsInCartesian` is never replaced.


In [ ]:

convFromRealToVirtual = realToVirtualEarthCoords(
    boxGrids.allGridsInGeoPoints,
    northPolePoint,
    northPoleInVirtualEarth;
    source_planet=:Earth,
    target_planet=:SphericalEarth,
    plane=:xz,
)
@unpack convertedGrids, rotationAngles, effectiveCoords = convFromRealToVirtual

newBoxGridsInVirtualEarth = (;
    boxGrids...,
    allGridsInGeoPoints=convertedGrids,
)



In [ ]:
# Optional checks for the rotated 2D disk.
@show extrema(p.lon for p in convertedGrids)
@show extrema(p.ecef[2] for p in convertedGrids)
@show rotationAngles.Δθ rotationAngles.Δφ # degrees, for diagnostics
@show rotationAngles.θshift rotationAngles.ϕshift # radians, used by StagYY


In [ ]:
# Detector position in the original regular meridional disk.
modifiedLongitude2Disk = 0.0 # use 0.0 or 180.0 for the mirrored disk
@show detectorPointFlat = GeoPoint(
    detectorPoint.lat,
    modifiedLongitude2Disk;
    alt=detectorPoint.alt,
)
detectorLocal = SVector(detectorPointFlat.ecef[1], detectorPointFlat.ecef[3])


#### Regular plotting/solver coordinates

For a 2D model, `allGridsInCartesian` contains `(x,z)` coordinates in metres. For 3D it contains `(x,y,z)`. These coordinates remain regular after creating the separate effective StagYY lookup coordinates.


In [ ]:
Nx, Nz = size(seismicModel.ρ)
xvals = [p.xz[1] for p in boxGrids.allGridsInCartesian[:, 1]] .* 1e-3
zvals = [p.xz[2] for p in boxGrids.allGridsInCartesian[1, :]] .* 1e-3

figSeismic = Figure(size=(850, 700))
axSeismic = Axis(
    figSeismic[1, 1];
    aspect=DataAspect(),
    xlabel="horizontal [km]",
    ylabel="vertical [km]",
    title="Ellipsoidal seismic model with topography",
)
hmSeismic = heatmap!(
    axSeismic,
    xvals,
    zvals,
    seismicModel.ρ;
    colormap=:turbo,
    colorrange=(0, 14),
)
scatter!(
    axSeismic,
    [detectorLocal[1] * 1e-3],
    [detectorLocal[2] * 1e-3];
    marker=:star5,
    markersize=24,
    color=:yellow,
    strokecolor=:black,
    strokewidth=1.5,
)
Colorbar(figSeismic[1, 2], hmSeismic; label="density [g cm⁻³]")
figSeismic


## Earth model modification (with different 2D/3D images)

### Reading StagYY data on its native periodic disk

The default `interpolation_method=:polar` samples StagYY directly on its structured periodic `(angle, radius)` grid. This avoids the Cartesian DIVAnd disk solve that can generate a radial segment at θ=0. The known StagYY seam itself is blended over a configurable ±2° wedge before sampling.

Use `interpolation_method=:divand` only as a fallback for experiments with the older Cartesian analysis. In that fallback, `analysis_size` remains automatic from `correlationLength` and `max_analysis_points` caps memory/CPU.


In [ ]:
correlationLength = (20e3, 20e3)

# The real 1D-model CMB is the default hybrid boundary. `effectiveRadii`
# expresses the ellipsoidal grid as equivalent radii in the 1D Earth model.
realCMBRadius = planet1D.my1DDSMmodel.averagedPlanetCMBInKilometer * 1e3
virtualSurfaceRadius = planet_ellipsoid(:SphericalEarth).a

ρCartesian = getCartesianField(
    ρGeodyFile,
    effectiveCoords.X,
    effectiveCoords.Y;
    rotationAngles=rotationAngles,
    interpolation_method=:polar,
    correlationLength=correlationLength,
    epsilon2=1.0,
    # These two options are used only by interpolation_method=:divand.
    # analysis_size is automatic from correlationLength.
    max_analysis_points=500_000,
    # Radially map the StagYY mantle onto the real CMB and spherical surface.
    target_cmb_radius=realCMBRadius,
    target_surface_radius=virtualSurfaceRadius,
    clamp_to_surface=true,
    # Remove the StagYY θ=0 radial artifact before DIVAnd (set 0 to disable).
    seam_blend_angle=deg2rad(0.0),
)

# StagYY density files are in kg m⁻³; the seismic model uses g cm⁻³.
stagyyDensityScale = 1e-3
ρStagYY = ρCartesian.field .* stagyyDensityScale
ρDiffStagYY = ρCartesian.diffField .* stagyyDensityScale

# This mask follows the real ellipsoidal CMB rather than StagYY's rcmb.
realEllipsoidalCMBMask = boxGrids.effectiveRadii .>= realCMBRadius
solidDensityThreshold = 1.0 # excludes water (1.0) and air (0.001)
stagyyMask = realEllipsoidalCMBMask .&
    (seismicModel.ρ .> solidDensityThreshold)

# Superpose StagYY in the real mantle/topographic solid. Keep the seismic
# model in the core, water, air, and outside the real topographic surface.
ρHybrid = copy(seismicModel.ρ)
ρHybrid[stagyyMask] .= ρStagYY[stagyyMask]

ρStagYYLayer = fill(NaN, size(ρHybrid))
ρStagYYLayer[stagyyMask] .= ρStagYY[stagyyMask]
ρDiffLayer = fill(NaN, size(ρHybrid))
ρDiffLayer[stagyyMask] .= ρDiffStagYY[stagyyMask]

diffColorLimit = maximum(abs, ρDiffStagYY[stagyyMask])
figHybrid = Figure(size=(1250, 1050))
axSeismic = Axis(figHybrid[1, 1]; aspect=DataAspect(), title="Seismic + topography")
axStagYY = Axis(figHybrid[1, 2]; aspect=DataAspect(), title="StagYY mantle density")
axDiff = Axis(figHybrid[2, 1]; aspect=DataAspect(), title="StagYY Δρ = ρ - angular mean")
axHybrid = Axis(figHybrid[2, 2]; aspect=DataAspect(), title="Hybrid density model")

hmSeismic = heatmap!(axSeismic, xvals, zvals, seismicModel.ρ;
    colormap=:turbo, colorrange=(0, 14))
hmStagYY = heatmap!(axStagYY, xvals, zvals, ρStagYYLayer;
    colormap=:turbo, colorrange=(0, 14))
hmDiff = heatmap!(axDiff, xvals, zvals, ρDiffLayer;
    colormap=:balance, colorrange=(-diffColorLimit, diffColorLimit))
hmHybrid = heatmap!(axHybrid, xvals, zvals, ρHybrid;
    colormap=:turbo, colorrange=(0, 14))

for ax in (axSeismic, axStagYY, axDiff, axHybrid)
    ax.xlabel = "horizontal [km]"
    ax.ylabel = "vertical [km]"
end
scatter!(
    axHybrid,
    [detectorLocal[1] * 1e-3],
    [detectorLocal[2] * 1e-3];
    marker=:star5,
    markersize=20,
    color=:yellow,
    strokecolor=:black,
)
Colorbar(figHybrid[1, 3], hmHybrid; label="density [g cm⁻³]")
Colorbar(figHybrid[2, 3], hmDiff; label="Δρ [g cm⁻³]")
figHybrid


## just by curiosity let's have a look on T & C

In [ ]:
CCartesian = getCartesianField(
    geody_files.composition,
    effectiveCoords.X,
    effectiveCoords.Y;
    rotationAngles=rotationAngles,
    interpolation_method=:polar,
    correlationLength=correlationLength,
    epsilon2=1.0,
    # These two options are used only by interpolation_method=:divand.
    # analysis_size is automatic from correlationLength.
    max_analysis_points=500_000,
    # Radially map the StagYY mantle onto the real CMB and spherical surface.
    target_cmb_radius=realCMBRadius,
    target_surface_radius=virtualSurfaceRadius,
    clamp_to_surface=true,
    # Remove the StagYY θ=0 radial artifact before DIVAnd (set 0 to disable).
    seam_blend_angle=deg2rad(0.0),
)
TCartesian = getCartesianField(
    geody_files.temperature,
    effectiveCoords.X,
    effectiveCoords.Y;
    rotationAngles=rotationAngles,
    interpolation_method=:polar,
    correlationLength=correlationLength,
    epsilon2=1.0,
    # These two options are used only by interpolation_method=:divand.
    # analysis_size is automatic from correlationLength.
    max_analysis_points=500_000,
    # Radially map the StagYY mantle onto the real CMB and spherical surface.
    target_cmb_radius=realCMBRadius,
    target_surface_radius=virtualSurfaceRadius,
    clamp_to_surface=true,
    # Remove the StagYY θ=0 radial artifact before DIVAnd (set 0 to disable).
    seam_blend_angle=deg2rad(0.0),
)
CLayer = fill(NaN, size(ρHybrid))
CLayer[stagyyMask] .= CCartesian.field[stagyyMask]
TLayer = fill(NaN, size(ρHybrid))
TLayer[stagyyMask] .= TCartesian.field[stagyyMask]

figTCfield = Figure(size=(1250, 1050))
axTemperature = Axis(figTCfield[1, 1]; aspect=DataAspect(), title="temperature")
axComposition = Axis(figTCfield[1, 2]; aspect=DataAspect(), title="composition (basalt)")


hmTemperature = heatmap!(axTemperature, xvals, zvals, TLayer;
    colormap=:seismic,)
hmComposition = heatmap!(axComposition, xvals, zvals, CLayer;
    colormap=:viridis, )


for ax in (axSeismic, axStagYY, axDiff, axHybrid)
    ax.xlabel = "horizontal [km]"
    ax.ylabel = "vertical [km]"
end
scatter!(
    axTemperature,
    [detectorLocal[1] * 1e-3],
    [detectorLocal[2] * 1e-3];
    marker=:star5,
    markersize=20,
    color=:yellow,
    strokecolor=:black,
)
scatter!(
    axComposition,
    [detectorLocal[1] * 1e-3],
    [detectorLocal[2] * 1e-3];
    marker=:star5,
    markersize=20,
    color=:yellow,
    strokecolor=:black,
)
#Colorbar(figHybrid[1, 3], hmHybrid; label="density [g cm⁻³]")
#Colorbar(figHybrid[2, 3], hmDiff; label="Δρ [g cm⁻³]")
figTCfield


### Empirical layered Z/A model with rotated StagYY water

The dry radial model is configurable data rather than hard-coded branches. The real ellipsoidal equivalent radius selects inner core, outer core, and mantle. StagYY water is sampled with exactly the same rotation, periodic polar interpolation, and CMB-to-surface mapping as density, but is mixed only into the mantle. Ocean points use `Z/A(H₂O)=5/9`; air remains zero.


In [ ]:
realICBRadius = 1_221_500.0 # m; empirical default from the cited three-layer model
realSurfaceRadius = planet1D.my1DDSMmodel.averagedPlanetRadiusInKilometer * 1e3

zOverALayers = makeZOverALayers(
    icb_radius=realICBRadius,
    cmb_radius=realCMBRadius,
    surface_radius=realSurfaceRadius,
    inner_core=0.466,
    outer_core=0.466,
    mantle=0.496,
)

# Include the solid Earth and ocean in the final map, but apply StagYY's
# internal water fraction only inside the mantle.
earthMaterialMask = ρHybrid .> 0.01
mantleWaterMask = realEllipsoidalCMBMask .&
    (seismicModel.ρ .> solidDensityThreshold)

zOverAResult = getZOverAField(
    wGeodyFile,
    effectiveCoords.X,
    effectiveCoords.Y,
    boxGrids.effectiveRadii;
    layers=zOverALayers,
    water_scale=1.0,
    water_z_over_a=5 / 9,
    material_mask=earthMaterialMask,
    water_mask=mantleWaterMask,
    outside=0.0,
    rotationAngles=rotationAngles,
    interpolation_method=:polar,
    target_cmb_radius=realCMBRadius,
    target_surface_radius=virtualSurfaceRadius,
    seam_blend_angle=deg2rad(2.0),
)

ZOverADry = zOverAResult.dry
ZOverA = copy(zOverAResult.mixed)
waterFraction = zOverAResult.water_fraction

# Represent the real ocean/topographic water column as H₂O as well.
oceanMask = isapprox.(seismicModel.ρ, 1.0; atol=1e-6)
ZOverA[oceanMask] .= 5 / 9

figZA = Figure(size=(1450, 500))
axDry = Axis(figZA[1, 1]; aspect=DataAspect(), title="Dry layered Z/A")
axWater = Axis(figZA[1, 2]; aspect=DataAspect(), title="Rotated StagYY water fraction")
axZA = Axis(figZA[1, 3]; aspect=DataAspect(), title="Water-corrected Z/A")
hmDry = heatmap!(axDry, xvals, zvals, ZOverADry;
    colormap=:viridis, colorrange=(0.46, 0.56))
hmWater = heatmap!(axWater, xvals, zvals, waterFraction;
    colormap=:blues)
hmZA = heatmap!(axZA, xvals, zvals, ZOverA;
    colormap=:viridis, colorrange=(0.46, 0.56))
for ax in (axDry, axWater, axZA)
    ax.xlabel = "horizontal [km]"
    ax.ylabel = "vertical [km]"
end
Colorbar(figZA[1, 4], hmZA; label="Z/A")
figZA


### Electron-density proxy and effective-radius anomaly

Here `nₑ = ρ Z/A`. The radial reference is computed in annular bins of the real ellipsoidal `effectiveRadii`, using only Earth material so exterior air does not bias the near-surface average.


In [ ]:
nₑ = electronDensity(ρHybrid, ZOverA)

radialBinWidth = min(Δx, Δz)
electronDensityStats = radialAverageAnomaly(
    nₑ,
    boxGrids.effectiveRadii;
    bin_width=radialBinWidth,
    mask=earthMaterialMask,
    minimum_radius=0.0,
    outside=NaN,
)

nₑRadialMean = electronDensityStats.radial_mean
δnₑ = electronDensityStats.anomaly
δnₑColorLimit = maximum(abs, δnₑ[earthMaterialMask])

figNe = Figure(size=(1450, 500))
axNe = Axis(figNe[1, 1]; aspect=DataAspect(), title="nₑ = ρ Z/A")
axNeMean = Axis(figNe[1, 2]; aspect=DataAspect(), title="nₑ averaged on effective radius")
axDeltaNe = Axis(figNe[1, 3]; aspect=DataAspect(), title="δnₑ = nₑ - ⟨nₑ⟩ᵣ")
hmNe = heatmap!(axNe, xvals, zvals, nₑ; colormap=:turbo)
hmNeMean = heatmap!(axNeMean, xvals, zvals, nₑRadialMean; colormap=:turbo)
hmDeltaNe = heatmap!(
    axDeltaNe,
    xvals,
    zvals,
    δnₑ;
    colormap=:balance,
    colorrange=(-δnₑColorLimit*0.1, δnₑColorLimit*0.1),
)
scatter!(
    axDeltaNe,
    [detectorLocal[1] * 1e-3],
    [detectorLocal[2] * 1e-3];
    marker=:star5,
    markersize=24,
    color=:yellow,
    strokecolor=:black,
    strokewidth=1.5,
)
for ax in (axNe, axNeMean, axDeltaNe)
    ax.xlabel = "horizontal [km]"
    ax.ylabel = "vertical [km]"
end
Colorbar(figNe[1, 4], hmDeltaNe; label="δnₑ [g cm⁻³]")
figNe


# getting n_e !!

In [ ]:
# Sample nₑ along each incoming ray, from the real topographic surface to the detector.
# Coordinates and Δbaseline are in metres; returned segment baselines are in km.
Δbaseline = 100e3
xPathCoordinates = [point.xz[1] for point in boxGrids.allGridsInCartesian[:, 1]]
zPathCoordinates = [point.xz[2] for point in boxGrids.allGridsInCartesian[1, :]]

electronPathSampling = creationPaths(
    nₑ,
    xPathCoordinates,
    zPathCoordinates,
    detectorLocal,
    binning;
    Δbaseline=Δbaseline,
    material_mask=earthMaterialMask,
    modifiedLongitude2Disk=modifiedLongitude2Disk,
    center=(0.0, 0.0),
    boundary_search_step=min(Δx, Δz) / 2,
    boundary_tolerance=10.0,
)

@assert length(electronPathSampling.profiles) == length(cosθgrid)
@assert all(profile -> profile.source != profile.detector,
            electronPathSampling.profiles)
@show extrema(sum(profile.baselines) for profile in electronPathSampling.profiles)

# Geometry check: every displayed ray is ordered source → detector.
selectedPathIndices = unique(round.(Int, range(
    1,
    length(electronPathSampling.profiles);
    length=min(9, length(electronPathSampling.profiles)),
)))
figPaths = Figure(size=(1300, 550))
axPathMap = Axis(
    figPaths[1, 1];
    aspect=DataAspect(),
    xlabel="horizontal [km]",
    ylabel="vertical [km]",
    title="Topographic sources → detector",
)
heatmap!(axPathMap, xvals, zvals, nₑ; colormap=:turbo)
for pathIndex in selectedPathIndices
    profile = electronPathSampling.profiles[pathIndex]
    lines!(
        axPathMap,
        [profile.source[1], profile.detector[1]] .* 1e-3,
        [profile.source[2], profile.detector[2]] .* 1e-3;
        linewidth=1.5,
    )
    scatter!(
        axPathMap,
        [profile.source[1] * 1e-3],
        [profile.source[2] * 1e-3];
        markersize=7,
        color=:white,
    )
end
scatter!(
    axPathMap,
    [detectorLocal[1] * 1e-3],
    [detectorLocal[2] * 1e-3];
    marker=:star5,
    markersize=22,
    color=:yellow,
    strokecolor=:black,
)

axPathProfiles = Axis(
    figPaths[1, 2];
    xlabel="distance from source [km]",
    ylabel="segment-averaged nₑ",
    title="Source → detector profiles",
)
for pathIndex in selectedPathIndices
    profile = electronPathSampling.profiles[pathIndex]
    distanceMidpoints = cumsum(profile.baselines) .- profile.baselines ./ 2
    lines!(
        axPathProfiles,
        distanceMidpoints,
        profile.values;
        label="cosθ=$(round(profile.cosθ; digits=2))",
    )
end
axislegend(axPathProfiles; position=:rt)
figPaths


In [ ]:
paths=electronPathSampling.paths
probabilities=Pνν(Uround, Hround, energies, paths)


##

## neutrino flux (completely new!!)


In [ ]:
energies = logrange(minEnergy, maxEnergy, numberEnergyBins)
# read the neutrino-flux table nuflux.csv given in .../Neutrino-Flux/data
bin_centers, flux_νe_interp, flux_νμ_interp, flux_antiνe_interp, flux_antiνμ_interp, energies, flux_νe, flux_νμ, flux_antiνe, flux_antiνμ = read_neutrino_flux_table("nuflux", numberEnergyBins, numberAngleBins, false; data_dir="../dataInput/neutrinoFluxData/")


In [ ]:
 # otherwise it will be daemon flux; this thing should be given in neutrinoFlux/cofig.jl

In [ ]:
# neutrino fluxes look good
# (comparing with Figure 3 in arXiv:1502.03916;
# there are slight differences, especially at low energies, because
# daemonflux is a bit different from honda)

numberEnergyBinSegments = numberEnergyBins - 1 

p1 = plot(energies, (energies.^3) .* sum(flux_νe,     dims=2) ./ numberEnergyBinSegments, label=L"$\nu_e$", 
          xaxis=:log, yaxis=:log, legend=:bottom,
          xlabel="E/GeV", ylabel=L"$\phi \times E_\nu^3$ /(m$^{-2}$s$^{-1}$sr$^{-1}$GeV$^{2}$")
plot!(p1, energies, (energies.^3) .* sum(flux_νμ,     dims=2) ./ numberEnergyBinSegments, label=L"$\nu_\mu$")
plot!(p1, energies, (energies.^3) .* sum(flux_antiνe, dims=2) ./ numberEnergyBinSegments, label=L"$\bar{\nu}_e$")
plot!(p1, energies, (energies.^3) .* sum(flux_antiνμ, dims=2) ./ numberEnergyBinSegments, label=L"$\bar{\nu}_\mu$")


In [ ]:
# linear interpolation on the flux dimension vs. correctly interpolated flux (in log of flux) 
# but anyways, it works (< 0.2%)

# plot the differences between the original and the interpolated fluxes
# to make sure that the interpolation is working correctly
# -> for the original flux we take the mean values
flux_νe_original     = 0.5 .* (sum(flux_νe,     dims=2)[1:end-1] .+ sum(flux_νe, dims=2)[2:end])
flux_νμ_original     = 0.5 .* (sum(flux_νμ,     dims=2)[1:end-1] .+ sum(flux_νμ, dims=2)[2:end])
flux_antiνe_original = 0.5 .* (sum(flux_antiνe, dims=2)[1:end-1] .+ sum(flux_antiνe, dims=2)[2:end])
flux_antiνμ_original = 0.5 .* (sum(flux_antiνμ, dims=2)[1:end-1] .+ sum(flux_antiνμ, dims=2)[2:end])
p2 = plot(bin_centers, 100 .* (sum(flux_νe_interp, dims=2) .- flux_νe_original) ./ flux_νe_original, 
          label=L"$\nu_e$", xlabel="E/GeV", ylabel="100 * (original - interpolation) / interpolation")
plot!(p2, bin_centers, 100 .* (sum(flux_νμ_interp,     dims=2) .- flux_νμ_original)     ./ flux_νμ_original,     label=L"$\nu_\mu$") 
plot!(p2, bin_centers, 100 .* (sum(flux_antiνe_interp, dims=2) .- flux_antiνe_original) ./ flux_antiνe_original, label=L"$\bar{\nu}_e$") 
plot!(p2, bin_centers, 100 .* (sum(flux_antiνμ_interp, dims=2) .- flux_antiνμ_original) ./ flux_antiνμ_original, label=L"$\bar{\nu}_\mu$") 


In [ ]:
size(flux_νe)

# i don't touch the pb of reverse(')

In [ ]:

# This is what we need to compute the interacting events!:
# TO-DO: put this computation in the function, no user should be doing this XD
flux_νe = reverse(flux_νe', dims=1)
flux_νμ = reverse(flux_νμ', dims=1)
flux_antiνe = reverse(flux_antiνe', dims=1)
flux_antiνμ = reverse(flux_antiνμ', dims=1)

In [ ]:
# neutrino oscillation probabilities look good
# TO-DO: Paνe2aντ deserves some attention → do a difference with OscProb!
minX, maxX, nX = -6500e3, 6500e3, 521
n_angles  = 100
n_pts     = 100
zposition = 2.5e3
energy_min = 1.0
energy_max = 100.0
energy_max = 40.0
osc_probs_nu, osc_probs_antinu = Neutrino_Oscillations.produce_neutrino_oscillation_probabilities(minX, maxX, nX, n_angles, n_pts, zposition, energy_min=energy_min, energy_max=energy_max)
# neutrinos, NMO
Pνe2νe = osc_probs_nu[:, :, 1, 1]' 
Pνe2νμ = osc_probs_nu[:, :, 1, 2]' 
Pνe2ντ = osc_probs_nu[:, :, 1, 3]' 
Pνμ2νe = osc_probs_nu[:, :, 2, 1]' 
Pνμ2νμ = osc_probs_nu[:, :, 2, 2]' 
Pνμ2ντ = osc_probs_nu[:, :, 2, 3]' 
# antineutrinos, NMO
Paνe2aνe = osc_probs_antinu[:, :, 1, 1]' 
Paνe2aνμ = osc_probs_antinu[:, :, 1, 2]' 
Paνe2aντ = osc_probs_antinu[:, :, 1, 3]' 
Paνμ2aνe = osc_probs_antinu[:, :, 2, 1]' 
Paνμ2aνμ = osc_probs_antinu[:, :, 2, 2]' 
Paνμ2aντ = osc_probs_antinu[:, :, 2, 3]' 

# plots for neutrinos, NMO
angles = range(-1, 0, length=100)
titles = (
    L"P($\nu_e \rightarrow \nu_e$)", L"P($\nu_e \rightarrow \nu_\mu$)", L"P($\nu_e \rightarrow \nu_\tau$)",
    L"P($\nu_\mu \rightarrow \nu_e$)", L"P($\nu_\mu \rightarrow \nu_\mu$)", L"P($\nu_\mu \rightarrow \nu_\tau$)"
)
all_probs = (
    Pνe2νe, Pνe2νμ, Pνe2ντ,
    Pνμ2νe, Pνμ2νμ, Pνμ2ντ
)
plot_list = []
for idx in 1:6
    # determine grid positions for labels
    is_bottom_row = idx > 3
    is_left_column = (idx == 1 || idx == 4)
    is_right_column = (idx ==3 || idx == 6)
    p = heatmap(
        energies, angles, all_probs[idx],              
        title = titles[idx],
        xlabel = is_bottom_row ? "Energy/GeV" : "",
        ylabel = is_left_column ? L"\cos\theta" : "",
        colorbar_title = is_right_column ? "Oscillation probability" : "",
        xscale = :log10,
        clim = (0, 1),
        cmap = :viridis
    )
    push!(plot_list, p)
end
final_plot = plot(
    plot_list..., 
    layout = (2, 3), 
    size = (1000, 400), 
    margin = 1Plots.mm,
)

# plots for antineutrinos, NMO
titles = (
    L"P($\bar{\nu}_e \rightarrow \bar{\nu}_e$)", L"P($\bar{\nu}_e \rightarrow \bar{\nu}_\mu$)", L"P($\bar{\nu}_e \rightarrow \bar{\nu}_\tau$)",
    L"P($\bar{\nu}_\mu \rightarrow \bar{\nu}_e$)", L"P($\bar{\nu}_\mu \rightarrow \bar{\nu}_\mu$)", L"P($\bar{\nu}_\mu \rightarrow \bar{nu}_\tau$)"
)
all_probs = (
    Paνe2aνe, Paνe2aνμ, Paνe2aντ,
    Paνμ2aνe, Paνμ2aνμ, Paνμ2aντ
)
plot_list = []
for idx in 1:6
    # determine grid positions for labels
    is_bottom_row = idx > 3
    is_left_column = (idx == 1 || idx == 4)
    is_right_column = (idx ==3 || idx == 6)
    p = heatmap(
        energies, angles, all_probs[idx],              
        title = titles[idx],
        xlabel = is_bottom_row ? "Energy/GeV" : "",
        ylabel = is_left_column ? L"\cos\theta" : "",
        colorbar_title = is_right_column ? "Oscillation probability" : "",
        xscale = :log10,
        clim = (0, 1),
        cmap = :viridis
    )
    push!(plot_list, p)
end
final_plot = plot(
    plot_list..., 
    layout = (2, 3), 
    size = (1000, 400), 
    margin = 1Plots.mm
)
